In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
food_path = os.path.join(path, 'Q1_data.csv')
df_food = pd.read_csv(food_path)

In [ ]:
# Task 2: Write your code here:
df_food.head(1)

In [ ]:
# Task 3: Write your code here:
df_food.info()

In [ ]:
# Task 4: Write your code here:
df_food.describe()

In [ ]:
# Task 5: Write your code here:
plt.figure(figsize=(10, 5))
plt.hist(df_food['Delivery_Time'].dropna(), bins=50, edgecolor='black')
plt.title('delivery time Distribution')
plt.xlabel('time')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Task 1: Write your code here:
cols = ['Distance_km', 'Weather', 'Traffic_Level', 'Time_of_Day', 'Vehicle_Type', 'Preparation_Time_min', 'Courier_Experience_yrs', 'Delivery_Time']
df_clean = df_food[cols].copy()



In [ ]:
df_clean.info()

In [ ]:
# Task 2: Write your code here:
# Fill categorical columns with 'unknown' - missing likely means "not specified"
for col in ['Weather', 'Traffic_Level', 'Time_of_Day']:
    df_clean[col] = df_clean[col].fillna('unknown')

for colm in ['Courier_Experience_yrs', 'Delivery_Time']:
    df_clean[colm] = df_clean[colm].fillna(df_clean[colm].mean())

In [ ]:
# Task 3: Write your code here:
#  Do we have duplicate samples?
def check_duplicates(df_clean):
  duplicates = df_clean.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df_clean.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")
check_duplicates(df_clean)


In [ ]:
# Task 4: Write your code here:
from sklearn.preprocessing import OneHotEncoder #import OneHotEncoder

categorical_columns = ['Weather', 'Traffic_Level', 'Time_of_Day', 'Vehicle_Type']
categories = np.array(categorical_columns).reshape(-1, 1)

print('data before encoding:\n', categorical_columns) #show before encoding

# sparse_output=False Return a normal NumPy array instead of a sparse matrix
onehot_encoder = OneHotEncoder(sparse_output=False) # Instantiate OneHotEncoder

data_onehot_encoded = onehot_encoder.fit_transform(categories) # Apply fit_transform to the copied

print('\nData after encoding:\n', data_onehot_encoded) #show after encoding


In [ ]:
# Task 5: Write your code here:
from sklearn.preprocessing import StandardScaler
feature_cols = data_onehot_encoded + ['Distance_km', 'Preparation_Time_min', 'Courier_Experience_yrs']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(feature_cols)




In [ ]:
# Task 6: Write your code here:

plt.figure(figsize=(10, 5))
plt.hist(df_food['Delivery_Time'].dropna(), bins=50, edgecolor='black')
plt.title('delivery time Distribution')
plt.xlabel('time')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Task 1: Write your code here:
X = df_clean[feature_cols]
y = df_clean['Delivery_Time']

In [ ]:
# Task 2,3,4,5: Write your code here:
from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error


rf = RandomForestRegressor(n_estimators=200, max_depth=15,
                               class_weight='balanced', random_state=42)


# Define K-Fold Cross Validation
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Iterate through folds
for fold, (train_idx, test_idx) in enumerate(kf.split(X), start=1):
    # indexing for each fold
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    # print shapes
    print(f"Fold {fold}")
    print("  X_train shape:", X_train.shape)
    print("  X_test shape :", X_test.shape)
    print("  y_train shape:", y_train.shape)
    print("  y_test shape :", y_test.shape)
    print("-" * 30)




        # Train and predict
    rf.fit(X_train, y_train)
    y_pred = rf.predict(X_test)

    # Calculate metrics
    mae_scores= mean_absolute_error(y_test, y_pred)
    print(f"MAE score:  ${mae_scores:,.2f}")
    print(f"Average of MAE:  ${mae_scores.mean():,.2f}")

In [ ]:
# Task 1: Write your code here:
coeffs = {}

coeffs['Lasso'] = rf['LASSO Regression'].coef_
coeffs['Ridge'] = rf['Ridge Regression'].coef_

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
axes = axes.flatten()
features = X.columns

for i, (rf, coef) in enumerate(coeffs.items()):
  # Sort features by absolute coefficient value
  absolute_coef = np.abs(coef)
  sorted_idx = np.argsort(absolute_coef)

  ax = axes[i]
  ax.barh(features[sorted_idx], coef[sorted_idx])
  ax.set_title(f"{rf} Coefficients")
  ax.set_xlabel("Coefficient Value (Impact)")

plt.tight_layout()
plt.show()

In [ ]:
# Task 2: Write your code here:

plt.figure(figsize=(10, 5))
plt.hist(df_food['Delivery_Time'].dropna(), bins=30, edgecolor='black')
plt.title('delivery time Distribution')
plt.xlabel('time')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Task Bonus: Write your code here:
from sklearn.svm import SVR
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_squared_error as sklearn_mse, mean_absolute_error, r2_score

models = {
  "Support Vector Machine": SVR(kernel='rbf'),
  "LightGBM": LGBMRegressor(verbose=-1),

}

# Storage for results
all_results = {}

for name in models:
  all_results[name] = {'mse': [], 'rmse': [], 'r2': []}

n_splits = 5
kf = KFold(n_splits=5, shuffle=True, random_state=42)
for fold_idx, (train_index, test_index) in enumerate(kf.split(X)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  for model_name, model in models.items():
    print(f"Training {model_name}...")

    # Train
    model.fit(X_train, y_train)

    # Predict
    y_pred = model.predict(X_test)

    # Calculate metrics
    mse = sklearn_mse(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_pred)

    # Store results
    all_results[model_name]["mse"].append(mse)
    all_results[model_name]["rmse"].append(rmse)
    all_results[model_name]["r2"].append(r2)

        # Calculate metrics
    mae_scores= mean_absolute_error(y_test, y_pred)
    print(f"MAE score:  ${mae_scores:,.2f}")
    print(f"Average of MAE:  ${mae_scores.mean():,.2f}")